# Cellblock on Google Colab

This notebook runs more prisons from [LLM-Stanford-Prison-Experiment](https://github.com/pragyaangaur/LLM-Stanford-Prison-Experiment) on one Colab GPU. It uses the same `kaggle/runner.py` and the same `cellblock/sim.py` as the Kaggle runs.

1. Choose **Runtime > Change runtime type > GPU** (an L4 or A100 if you have them, a T4 otherwise).
2. Choose **Runtime > Run all** and allow access to Google Drive when asked.

Records are written to `MyDrive/cellblock/results/<tag>/runs` after every step. If the session drops, run all again: seeds that finished are skipped, and a seed with an unfinished prison is run again from day 1.

On a GPU with 20 GB or more the model runs in float16, which matches the Kaggle runs. On a T4 it runs in 4-bit NF4, which is a different build of the model, so those results are kept under their own tag and are never pooled with the Kaggle runs. Everything run here is exploratory. The preregistered confirmatory run is the Kaggle run on seeds 2012 to 2047.

In [ ]:
# What to run. Each plan block is variant:first_seed:number_of_seeds, and every seed runs as coached and as neutral.
# The default is the second model family named in PREREGISTRATION.md, on the same seeds as the exploratory Qwen run.
MODEL = "allenai/OLMo-2-1124-7B-Instruct"
PLAN = "calm:2000:12"
HOURS = 11.5   # stop starting new prisons after this long; Colab ends free sessions at 12 hours or earlier

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > GPU, then run all again."
props = [torch.cuda.get_device_properties(i) for i in range(torch.cuda.device_count())]
mem = sum(p.total_memory for p in props)
QUANT = "fp16" if mem >= 20e9 else "nf4"
MAX_WAVE = 12 if QUANT == "fp16" else 6   # smaller waves on a T4 lose less when a session drops
TAG = "colab-" + MODEL.split("/")[-1].lower() + "-" + QUANT
print(", ".join(p.name for p in props), f"{mem / 1e9:.0f} GB, so {QUANT}. Results tag: {TAG}")

from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/cellblock"

!rm -rf /content/repo && git clone -q --depth 1 https://github.com/pragyaangaur/LLM-Stanford-Prison-Experiment /content/repo
if QUANT == "nf4":
    !pip install -q bitsandbytes

In [ ]:
!cd /content/repo && python kaggle/build.py --plan {PLAN} --model {MODEL} --hours {HOURS} --tag {TAG} --quant {QUANT} --max-wave {MAX_WAVE}
!cd /content/repo && CELLBLOCK_WORK={WORK} python kaggle/push/cellblock_kaggle.py

Summarise what has finished so far and download it as a zip. Send the zip back to be added to the repository under `results/<tag>/`.

In [ ]:
!mkdir -p /content/repo/results && ln -sfn {WORK}/results/{TAG} /content/repo/results/{TAG}
!cd /content/repo && python scripts/analyze.py --tag {TAG} | head -60
!cd {WORK}/results && zip -qr /content/{TAG}.zip {TAG}
from google.colab import files
files.download(f"/content/{TAG}.zip")